# 1. Pipeline de visión paso a paso

Recorrido de la Fase 1 sobre una imagen: de la foto al JSON que recibe el solver. Cada paso llama a las funciones de `src/vision/` (el notebook no reimplementa nada). Detalles y decisiones en `docs/fase1_vision.md`.

Por defecto usa una imagen sintética que simula una foto. Para usar una foto real, cambiar `IMAGE` por su ruta.

In [ ]:
import os, sys
from pathlib import Path
# ejecutar desde la raíz del repositorio para que funcionen los imports de src/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

def show(*imgs, titles=None, size=4.5, cmap="gray"):
    fig, axes = plt.subplots(1, len(imgs), figsize=(size * len(imgs), size))
    axes = np.atleast_1d(axes)
    for k, (ax, im) in enumerate(zip(axes, imgs)):
        ax.imshow(im, cmap=cmap if im.ndim == 2 else None)
        ax.set_title(titles[k] if titles else "", fontsize=10, loc="left")
        ax.axis("off")
    plt.tight_layout(); plt.show()

In [ ]:
from src.eval.render import FONTS_TEST, available_fonts, make_sample
from src.vision.preprocess import load_gray, normalize_size, enhance, binarize

IMAGE = None            # p. ej. "data/raw/k01.jpg"
if IMAGE:
    original = load_gray(IMAGE); truth = None
else:
    original, truth = make_sample(3, 1003, available_fonts(FONTS_TEST))
show(original, titles=["Imagen de entrada"], size=6)

## Pasos 1–2: normalización, CLAHE y umbral adaptativo

El umbral adaptativo (y no Otsu global) tolera el gradiente de iluminación y las sombras. Queda invertido: líneas y tinta en blanco.

In [ ]:
gray, scale = normalize_size(original)
enhanced = enhance(gray)
binary = binarize(enhanced)
show(gray, enhanced, binary, titles=["Normalizada", "CLAHE + Gauss", "Umbral adaptativo (invertido)"])

## Pasos 3–4: esquinas y corrección de perspectiva

Se buscan contornos que, aproximados, sean cuadriláteros convexos grandes, y se elige el **más pequeño** (en una foto, la hoja de papel también es un cuadrilátero que contiene la grilla). Con las 4 esquinas se calcula la homografía `H`.

In [ ]:
import cv2
from src.vision.locate import find_corners, warp

corners, method = find_corners(binary)
vis = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
cv2.polylines(vis, [corners.astype(int)], True, (235, 104, 52), 4)
for (x, y) in corners.astype(int):
    cv2.circle(vis, (x, y), 12, (42, 120, 214), -1)
warped, H = warp(gray, corners)
print("método:", method); print("H =\n", np.round(H, 3))
show(vis, warped, titles=["Esquinas detectadas", "Vista rectificada"], size=5.5)

## Paso 5: tamaño de la grilla con perfiles de proyección

Una apertura morfológica deja solo los trazos largos; la suma por filas/columnas da picos en las líneas. Se prueba cada número de celdas `n` y se puntúa el perfil en `k·L/n`.

In [ ]:
from src.vision.grid import detect_grid, line_profiles, estimate_count, _score

prof_y, prof_x = line_profiles(warped)
grid = detect_grid(warped)
print(f"grilla detectada: {grid.rows} x {grid.cols}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
axes[0].plot(prof_x, color="#2a78d6", lw=1.2)
for x in grid.xs: axes[0].axvline(x, color="#eb6834", lw=0.8, alpha=0.6)
axes[0].set_title("Perfil de líneas verticales y líneas ajustadas", fontsize=10, loc="left")
ns = range(3, 41)
L = len(prof_x)
axes[1].bar(ns, [_score(prof_x, n, max(1, int(0.04 * L / n))) for n in ns], color="#2a78d6")
axes[1].set_title("Puntuación por número de columnas", fontsize=10, loc="left"); axes[1].set_xlabel("n")
plt.tight_layout(); plt.show()

## Pasos 6–7: celdas blancas y de pista

Dos medidas por celda: brillo relativo a la iluminación estimada (celdas oscuras) y contraste de la diagonal. Luego la **estructura** decide qué pistas existen: hay suma horizontal si la celda de la derecha es blanca, vertical si la de abajo lo es.

In [ ]:
from src.vision.cells import cell_features, classify, infer_clues

feats = cell_features(warped, grid)
white = classify(feats)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].scatter(feats.relative[white], feats.diagonal[white], s=14, color="#2a78d6", label="blanca")
axes[0].scatter(feats.relative[~white], feats.diagonal[~white], s=14, color="#eb6834", marker="s", label="pista")
axes[0].axvline(0.6, color="#999", ls="--", lw=0.8); axes[0].axhline(0.22, color="#999", ls="--", lw=0.8)
axes[0].set_xlabel("brillo relativo"); axes[0].set_ylabel("contraste de la diagonal"); axes[0].legend(frameon=False)
axes[0].set_title("Características de cada celda", fontsize=10, loc="left")
axes[1].imshow(white, cmap="gray"); axes[1].set_title("Clasificación (blanco = celda blanca)", fontsize=10, loc="left")
axes[1].axis("off"); plt.tight_layout(); plt.show()

## Pasos 8–10: triángulos, dígitos y CNN

Cada pista se recorta con una máscara triangular, se normaliza la polaridad (trazo claro sobre fondo oscuro) y se separan los dígitos. Cada dígito, centrado en 28×28, pasa por la CNN.

In [ ]:
from src.vision.digits import extract_region, segment_digits, to_canvas, min_digits_for
from src.vision.ocr_cnn import DigitReader

reader = DigitReader()
structure = infer_clues(white)
examples = [(i, j, d) for i in range(grid.rows) for j in range(grid.cols)
            if structure[i][j]["type"] == "clue" for d in ("right", "down") if structure[i][j][d] is not None][:4]
fig, axes = plt.subplots(len(examples), 4, figsize=(9, 2.3 * len(examples)))
for row, (i, j, d) in zip(axes, examples):
    x0, y0, x1, y1 = grid.cell_box(i, j)
    norm, mask = extract_region(warped, grid.cell_box(i, j), d)
    boxes = segment_digits(norm, mask)
    canv = [to_canvas(norm, b) for b in boxes]
    probs = reader.probabilities(canv)
    row[0].imshow(warped[y0:y1, x0:x1], cmap="gray"); row[0].set_title(f"celda ({i},{j}) {d}", fontsize=9, loc="left")
    row[1].imshow(norm, cmap="gray"); row[1].set_title("triángulo normalizado", fontsize=9, loc="left")
    for k in range(2):
        if k < len(canv):
            row[2 + k].imshow(canv[k], cmap="gray")
            row[2 + k].set_title(f"CNN: {probs[k].argmax()} ({probs[k].max():.2f})", fontsize=9, loc="left")
    for ax in row: ax.axis("off")
plt.tight_layout(); plt.show()

## Paso 11–12: validación y JSON

`process_gray` ejecuta todos los pasos anteriores, valida cada número con las reglas de Kakuro y genera el JSON (formato de la sección 3 de `docs/fase1_vision.md`).

In [ ]:
import json
from src.vision.pipeline import process_gray
from src.solver.__main__ import render
from src.solver.parse import parse_puzzle
from src.solver.solve import SolveResult

puzzle = process_gray(original, reader).puzzle
print(render(parse_puzzle(puzzle), SolveResult("-", "-")))
print("\npistas inciertas:", puzzle["uncertain_clues"])
print(json.dumps({k: puzzle[k] for k in ("rows", "cols")} | {"grid[1]": puzzle["grid"][1][:3]}, indent=1))
if truth is not None:
    ok = all(puzzle["grid"][i][j] == truth["grid"][i][j] for i in range(truth["rows"]) for j in range(truth["cols"]))
    print("\n¿coincide con la etiqueta real?", ok)